# 01 EAR and MAR Exploration

This notebook explores:
- Eye Aspect Ratio (EAR)
- Mouth Aspect Ratio (MAR)
- dataset availability in `data/`
- model asset availability in `models/`


In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple
import math

import cv2
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'

OPEN_DIR = DATA_DIR / 'open'
CLOSED_DIR = DATA_DIR / 'closed'
SESSIONS_DIR = DATA_DIR / 'sessions'

EYE_MODEL_PATH = MODELS_DIR / 'eye_classifier.h5'
FACE_LANDMARKER_MODEL_PATH = MODELS_DIR / 'face_landmarker.task'


In [ ]:
def file_summary(directory: Path) -> Dict[str, object]:
    files = [path for path in directory.glob('*') if path.is_file()]
    return {
        'directory': str(directory),
        'exists': directory.exists(),
        'count': len(files),
        'examples': [path.name for path in files[:5]],
    }


data_summary = {
    'open': file_summary(OPEN_DIR),
    'closed': file_summary(CLOSED_DIR),
    'sessions': file_summary(SESSIONS_DIR),
}

model_summary = {
    'eye_classifier_exists': EYE_MODEL_PATH.exists(),
    'eye_classifier_size': EYE_MODEL_PATH.stat().st_size if EYE_MODEL_PATH.exists() else 0,
    'face_landmarker_exists': FACE_LANDMARKER_MODEL_PATH.exists(),
    'face_landmarker_size': FACE_LANDMARKER_MODEL_PATH.stat().st_size if FACE_LANDMARKER_MODEL_PATH.exists() else 0,
}

print('Data summary:')
for name, summary in data_summary.items():
    print(name, summary)

print('\nModel summary:')
for name, value in model_summary.items():
    print(name, value)


In [ ]:
def euclidean_distance(point_a: Tuple[int, int], point_b: Tuple[int, int]) -> float:
    return math.dist(point_a, point_b)


def calculate_ear(eye_points: List[Tuple[int, int]]) -> float:
    vertical_1 = euclidean_distance(eye_points[1], eye_points[5])
    vertical_2 = euclidean_distance(eye_points[2], eye_points[4])
    horizontal = euclidean_distance(eye_points[0], eye_points[3])
    if horizontal == 0:
        return 0.0
    return (vertical_1 + vertical_2) / (2.0 * horizontal)


def calculate_mar(mouth_points: List[Tuple[int, int]]) -> float:
    vertical_1 = euclidean_distance(mouth_points[1], mouth_points[5])
    vertical_2 = euclidean_distance(mouth_points[2], mouth_points[4])
    horizontal = euclidean_distance(mouth_points[0], mouth_points[3])
    if horizontal == 0:
        return 0.0
    return (vertical_1 + vertical_2) / (2.0 * horizontal)


In [ ]:
sample_open_eye = [(0, 10), (10, 4), (20, 3), (40, 10), (20, 17), (10, 16)]
sample_closed_eye = [(0, 10), (10, 9), (20, 9), (40, 10), (20, 11), (10, 11)]
sample_closed_mouth = [(0, 15), (10, 12), (20, 13), (40, 15), (20, 17), (10, 18)]
sample_open_mouth = [(0, 15), (10, 5), (20, 6), (40, 15), (20, 24), (10, 25)]

print('Sample open-eye EAR:', round(calculate_ear(sample_open_eye), 3))
print('Sample closed-eye EAR:', round(calculate_ear(sample_closed_eye), 3))
print('Sample closed-mouth MAR:', round(calculate_mar(sample_closed_mouth), 3))
print('Sample open-mouth MAR:', round(calculate_mar(sample_open_mouth), 3))


In [ ]:
def plot_landmark_shape(points: List[Tuple[int, int]], title: str, color: str) -> None:
    xs = [point[0] for point in points] + [points[0][0]]
    ys = [point[1] for point in points] + [points[0][1]]
    plt.plot(xs, ys, marker='o', color=color)
    plt.title(title)
    plt.gca().invert_yaxis()
    plt.axis('equal')
    plt.grid(True, alpha=0.3)


plt.figure(figsize=(10, 8))
plt.subplot(2, 2, 1)
plot_landmark_shape(sample_open_eye, 'Open Eye (EAR)', 'green')

plt.subplot(2, 2, 2)
plot_landmark_shape(sample_closed_eye, 'Closed Eye (EAR)', 'red')

plt.subplot(2, 2, 3)
plot_landmark_shape(sample_closed_mouth, 'Closed Mouth (MAR)', 'blue')

plt.subplot(2, 2, 4)
plot_landmark_shape(sample_open_mouth, 'Open Mouth (MAR)', 'orange')

plt.tight_layout()
plt.show()


In [ ]:
def load_grayscale_images(directory: Path, limit: int = 6) -> List[np.ndarray]:
    images: List[np.ndarray] = []
    for path in sorted(directory.glob('*')):
        if path.suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
            continue
        image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if image is not None:
            images.append(image)
        if len(images) >= limit:
            break
    return images


open_images = load_grayscale_images(OPEN_DIR)
closed_images = load_grayscale_images(CLOSED_DIR)
print('Loaded open previews:', len(open_images))
print('Loaded closed previews:', len(closed_images))


In [ ]:
def preview_image_group(images: List[np.ndarray], title_prefix: str) -> None:
    if not images:
        print(f'No images available for {title_prefix}.')
        return

    columns = min(3, len(images))
    rows = int(np.ceil(len(images) / columns))
    plt.figure(figsize=(12, 4 * rows))

    for index, image in enumerate(images, start=1):
        plt.subplot(rows, columns, index)
        plt.imshow(image, cmap='gray')
        plt.title(f'{title_prefix} {index}')
        plt.axis('off')

    plt.tight_layout()
    plt.show()


preview_image_group(open_images, 'Open eye')
preview_image_group(closed_images, 'Closed eye')


Use this notebook to confirm the project has enough `data/open` and `data/closed` images before training, and to reason about how EAR and MAR thresholds should behave.